###Setup

In [1]:
import urllib.request
import os
import re

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
file_path = "the-verdict.txt"

In [2]:
if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r', encoding='utf-8') as f1, open(file_path, 'r', encoding='utf-8') as f2:
    lines = f1.readlines()
    raw_text = f2.read()

###Simple Preprocess

In [5]:
# \s matches whitespace (spaces, tabs and new lines)
# () denotes a capturing group
# [] Square brackets to create a matching list that will match on any one of
# -- the characters in the list (only one).
# So this regex matches either comma or periods (only one of them), OR newlines
# Multiple stages to show the different types of splitting that we can do
def simple_preprocess(text: str, stage:int, keep_whitespaces:bool=False) -> str:
    if str(stage).isnumeric and len(str(stage)) == 1 and isinstance(stage, int):
        if stage == 1:
            # Only commas, periods, and whitespaces
            res = re.split(r'([,.]|\s)', text)
        elif stage == 2:
            # Comma, dot, colon, semicolon, question mark, underscore,
            # -- and exclamation point
            res = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        if not keep_whitespaces:
            res = [txt.strip() for txt in res if txt.strip()]
        return res
    else:
        return text

In [6]:
n = 50
print(f"Total amount of characters: {len(raw_text)} ")
print(f"Total amount of lines: {len(lines)}")
print(f"First line: {lines[0]}")
print(f"First {n} characters: {raw_text[:n]}")

preprocessed = simple_preprocess(raw_text, 2, False)
print(f"First {n} characters of the modified text: {preprocessed[:n]}")
print(f"Lenght: {len(preprocessed)}")

Total amount of characters: 20479 
Total amount of lines: 165
First line: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

First 50 characters: I HAD always thought Jack Gisburn rather a cheap g
First 50 characters of the modified text: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself']
Lenght: 4690


In [7]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocab size: {vocab_size}")

vocab = {token:id for id, token in enumerate(all_words)}
for idx, item in enumerate(vocab.items()):
    print(item)
    if idx >= 50:
        break

Vocab size: 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Simple Tokenizer Class

In [3]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # Vocab contains the dictionary mapping each word to its id
        self.str_to_int = vocab
        # Reverse mapping, from token id to the respective token
        self.int_to_str = {idx: tok for tok, idx in vocab.items()}

    def encode(self, text, keep_whitespaces: bool=False):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        if not keep_whitespaces:
            preprocessed = [
                item.strip() for item in preprocessed if item.strip()
            ]
        ids = [self.str_to_int[tok] for tok in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [8]:
# Preprocess first to get a vocabulary size
preprocessed_text = simple_preprocess(raw_text, 2, False)
all_tokens = sorted(set(preprocessed))
vocab = {token:id for id, token in enumerate(all_tokens)}
print(len(vocab.items()))

1130


In [9]:
SimpleTokV1 = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids = SimpleTokV1.encode(text)
print(f"Encoded text: {ids}")
print(f"Decoded ids: {SimpleTokV1.decode(ids)}")

Encoded text: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
Decoded ids: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


If we use a text such as this, we will get a `KeyError` "`cakes` is not present in the current vocabulary."

In [10]:
text ="""I like waffles"""
try:
    ids = SimpleTokV1.encode(text)
    print(ids)
    print(SimpleTokV1.decode(ids))
except KeyError as e:
    print(f"KeyError: the {e} word is not present in the current vocabulary.")

KeyError: the 'waffles' word is not present in the current vocabulary.


Therefore we will add <|unk|> and <|endoftext|> tokens

In [11]:
preprocessed_text = simple_preprocess(raw_text, 2, False)
all_tokens = sorted(list(set(preprocessed_text)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
extended_vocab = {token:id for id, token in enumerate(all_tokens)}
print(len(extended_vocab.items()))

# Last 5 elements
for key, value in enumerate(list(extended_vocab.items())[-5:]):
    print(key, value)

1132
0 ('younger', 1127)
1 ('your', 1128)
2 ('yourself', 1129)
3 ('<|endoftext|>', 1130)
4 ('<|unk|>', 1131)


####Update the Tokenizer Class

In [12]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # Vocab contains the dictionary mapping each word to its id
        self.str_to_int:dict = vocab
        # Reverse mapping, from token id to the respective token
        self.int_to_str:dict = {idx: tok for tok, idx in vocab.items()}

    def encode(self, text, keep_whitespaces: bool=False):
        preprocessed:list = re.split(r'([,.?_!"()\']|--|\s)', text)
        if not keep_whitespaces:
            preprocessed = [
                item.strip() for item in preprocessed if item.strip()
            ]
        # Check on the vocabulary keys (words)
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item
                        in preprocessed]
        ids = [self.str_to_int[tok] for tok in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

Trying it out

In [13]:
SimpleTokV2 = SimpleTokenizerV2(extended_vocab)

In [14]:
text1 = """I like waffles."""
text2 = """And I like ice cream!"""
text = "<|endoftext|>".join((text1, text2))
print(text)

I like waffles.<|endoftext|>And I like ice cream!


In [15]:
ids = SimpleTokV2.encode(text)
print(ids)
print(SimpleTokV2.decode(ids))

[53, 628, 1131, 7, 1131, 53, 628, 1131, 1131, 0]
I like <|unk|>. <|unk|> I like <|unk|> <|unk|>!


###Byte Pair Encoding

In [16]:
%pip install tiktoken
import tiktoken
print(f"Version: {tiktoken.__version__}")

Version: 0.12.0


In [17]:
tokenizer = tiktoken.get_encoding("gpt2")

In [18]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace"
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(f"Tokens: {integers}")

Tokens: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271]


In [19]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace


In [20]:
eof_tok = "<|endoftext|>"
print(f"the <|endoftext|> token is associated with the id {tokenizer.encode(eof_tok, allowed_special={"<|endoftext|>"})}")

the <|endoftext|> token is associated with the id [50256]


### Data sampling

In [21]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [22]:
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"x: {x}")
print(f"y: .... {y}")

x: [290, 4920, 2241, 287]
y: .... [4920, 2241, 287, 257]


In [23]:
for idx in range(1, context_size + 1):
    context = enc_sample[:idx]
    # What the model needs to predict
    target = enc_sample[idx]
    print(f"{context} ----> {target}")

print("\n")
for idx in range(1, context_size + 1):
    context = enc_sample[:idx]
    target = enc_sample[idx]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode([target])}")

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


Dataset custom class

In [24]:
import torch
from torch.utils.data import Dataset, DataLoader

# Inherits from base pytorch dataset class
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_lenght, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        # batch_size used later will be [id, ...], [target_id, ...]
        # start, stop, step
        # stride is the step
        # max_lenght: how many tokens in each chunk
        # stride: how much we shift at each iteration
        # it's how much we shift from batch to batch
        for idx in range(0, len(token_ids) - max_lenght, stride):
            input_chunk = token_ids[idx: idx + max_lenght]
            target_chunk = token_ids[idx + 1: idx + max_lenght + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [25]:
GPTDataset = GPTDatasetV1(raw_text, tokenizer, 4, 1)
print(GPTDataset[0])
print(GPTDataset[1])

(tensor([  40,  367, 2885, 1464]), tensor([ 367, 2885, 1464, 1807]))
(tensor([ 367, 2885, 1464, 1807]), tensor([2885, 1464, 1807, 3619]))


Function for creating a dataloader

In [26]:
def create_dataloader_v1(
        text,
        tokenizer,
        batch_size=4,
        max_length=256,
        stride=128,
        shuffle=True,
        drop_last=True,
        num_workers=0
):

    # batch_size = inputs + targets
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [27]:
dataloader = create_dataloader_v1(
    raw_text, tokenizer, batch_size=1,
    max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(f"First batch: {first_batch}")

First batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [28]:
second_batch = next(data_iter)
print(f"Second batch: {second_batch}")

Second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


Using a max_lenght of $4$ and stride of $4$ to avoid losing data.

In [29]:
dataloader = create_dataloader_v1(
    raw_text, tokenizer, batch_size=8,
    max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
print(f"First batch")

inputs, targets = next(data_iter)
print(f"Inputs: {inputs}")
print(f"Targets: {targets}")

First batch
Inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


###Creating Embeddings

In [31]:
# We need to convert the token ids into embeddings, floating point values
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [33]:
# batch_size of of 8, four tokens each, the result is an 8x4x256 tensor
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, tokenizer, batch_size=8,
    max_length=max_length,
    stride=max_length, shuffle=False,
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Token ids: {inputs}")
print(f"Inputs shape: {inputs.shape}")

Token ids: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Inputs shape: torch.Size([8, 4])


$8$ text samples with $4$ tokens each

In [34]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


Each token id is now a $256$ dimensional vector

####Positional Embedding

In [36]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [38]:
pos_embeddings

tensor([[-0.5078, -1.1024, -0.4172,  ..., -0.3074, -1.4610, -0.0541],
        [ 1.1283, -0.4921, -0.4066,  ..., -0.4692,  1.1133,  0.0390],
        [ 1.8324, -0.8049, -0.7988,  ..., -1.6644, -0.0351, -0.2605],
        [ 0.1467,  2.6398, -0.6718,  ..., -1.5684,  0.1755,  0.7793]],
       grad_fn=<EmbeddingBackward0>)

For every token in the $4$ tokens of each sample, we add a positional embedding $256$ dimensional vector

In [40]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
